# 🚀 Quick Start: Trajectory-Guided LTX Video Training on RunPod

This notebook provides a quick start guide to using the trajectory-guided LTX Video training system on RunPod.

## 📋 Prerequisites

- RunPod instance with GPU (A100, A6000, or better recommended)
- At least 100GB free disk space
- Internet connection for downloading models

## 🎯 What You'll Learn

1. Check environment setup
2. Download sample data
3. Extract trajectories
4. Visualize motion
5. Quick training test
6. Generate video

## Step 1: Environment Check

In [ ]:
import sys
import os

# Set workspace path
WORKSPACE = "/workspace/LTX_video_training"
sys.path.insert(0, WORKSPACE)
os.chdir(WORKSPACE)

print(f"Working directory: {os.getcwd()}")

In [ ]:
# Check PyTorch and CUDA
import torch
import subprocess

print("=" * 60)
print("System Information")
print("=" * 60)
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")
print(f"cuDNN version: {torch.backends.cudnn.version()}")
print(f"GPU count: {torch.cuda.device_count()}")

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"\nGPU {i}: {torch.cuda.get_device_name(i)}")
        print(f"  Memory: {torch.cuda.get_device_properties(i).total_memory / 1e9:.2f} GB")
        print(f"  Compute Capability: {torch.cuda.get_device_properties(i).major}.{torch.cuda.get_device_properties(i).minor}")

print("\n" + "=" * 60)

# Disk space
result = subprocess.run(['df', '-h', '/workspace'], capture_output=True, text=True)
print("Disk Usage:")
print(result.stdout)

In [ ]:
# Verify imports
print("Checking dependencies...")

try:
    import diffusers
    print(f"✓ diffusers: {diffusers.__version__}")
except ImportError as e:
    print(f"✗ diffusers: {e}")

try:
    import transformers
    print(f"✓ transformers: {transformers.__version__}")
except ImportError as e:
    print(f"✗ transformers: {e}")

try:
    import accelerate
    print(f"✓ accelerate: {accelerate.__version__}")
except ImportError as e:
    print(f"✗ accelerate: {e}")

try:
    import peft
    print(f"✓ peft: {peft.__version__}")
except ImportError as e:
    print(f"✗ peft: {e}")

try:
    import cv2
    print(f"✓ opencv: {cv2.__version__}")
except ImportError as e:
    print(f"✗ opencv: {e}")

print("\n✅ Environment check complete!")

## Step 2: Download Sample Video

Let's download a sample video to test trajectory extraction.

In [ ]:
# Download sample video from Pexels (free stock footage)
import urllib.request
from pathlib import Path

sample_dir = Path("./examples/sample_data")
sample_dir.mkdir(parents=True, exist_ok=True)

# Sample video URL (replace with actual URL)
sample_video_url = "https://example.com/sample.mp4"  # Update with actual URL
sample_video_path = sample_dir / "sample_video.mp4"

print("Note: Please upload your own sample video to:")
print(f"  {sample_video_path}")
print("\nOr download from Pexels/Pixabay and place in examples/sample_data/")

## Step 3: Extract Trajectories

Extract trajectory fields from the sample video.

In [ ]:
from src.preprocessing.trajectory_extractor import TrajectoryExtractor

# Initialize extractor
extractor = TrajectoryExtractor(
    device="cuda" if torch.cuda.is_available() else "cpu",
    enable_physics_smoothing=True
)

print("Extractor initialized!")

In [ ]:
# Extract trajectories (assumes you have a sample video)
if sample_video_path.exists():
    print(f"Extracting trajectories from {sample_video_path}...")
    
    trajectory_data = extractor.extract_from_video(
        video_path=str(sample_video_path),
        output_path=str(sample_dir / "trajectory_basic.mp4")
    )
    
    print(f"\nTrajectory shape: {trajectory_data['trajectories'].shape}")
    print(f"Confidence shape: {trajectory_data['confidence'].shape}")
    print(f"Occlusions detected: {trajectory_data['occlusions'].sum()}")
    print(f"\n✅ Trajectory extraction complete!")
else:
    print(f"⚠️  Sample video not found at {sample_video_path}")
    print("Please upload a video file first.")

## Step 4: Visualize Trajectories

Create different visualizations of the trajectory field.

In [ ]:
from src.preprocessing.trajectory_visualizer import TrajectoryVisualizer, ParticleTrajectoryVisualizer
import cv2
import matplotlib.pyplot as plt
from IPython.display import Video, display

if sample_video_path.exists() and 'trajectory_data' in locals():
    # Load first frame
    cap = cv2.VideoCapture(str(sample_video_path))
    ret, first_frame = cap.read()
    cap.release()
    first_frame = cv2.cvtColor(first_frame, cv2.COLOR_BGR2RGB)
    
    # Create visualizations
    visualizers = {
        'flow': TrajectoryVisualizer(visualization_type='flow'),
        'depth': TrajectoryVisualizer(visualization_type='depth'),
        'multi': TrajectoryVisualizer(visualization_type='multi'),
    }
    
    for name, visualizer in visualizers.items():
        print(f"Creating {name} visualization...")
        output_path = sample_dir / f"trajectory_{name}.mp4"
        
        vis_frames = visualizer.visualize(
            trajectory_data=trajectory_data,
            first_frame=first_frame,
            output_path=str(output_path)
        )
        
        print(f"  Saved: {output_path}")
    
    print("\n✅ Visualizations created!")
    
    # Display one visualization
    print("\nDisplaying multi-channel visualization:")
    display(Video(str(sample_dir / "trajectory_multi.mp4"), width=800))
else:
    print("⚠️  No trajectory data available. Run previous cells first.")

## Step 5: Check GPU Memory

Monitor GPU usage before training.

In [ ]:
import subprocess

print("Current GPU Status:")
print("=" * 80)
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

## 🎉 Quick Start Complete!

You've successfully:
- ✅ Verified the environment
- ✅ Extracted trajectories from video
- ✅ Created trajectory visualizations
- ✅ Checked GPU status

## Next Steps

1. **Dataset Preparation**: Open `02_Dataset_Preparation.ipynb`
2. **Training**: Open `03_Training.ipynb`
3. **Inference**: Open `04_Inference.ipynb`

## Need Help?

- Check `/workspace/LTX_video_training/README.md`
- Review `/workspace/LTX_video_training/docs/BEST_PRACTICES.md`
- Monitor training: `/workspace/LTX_video_training/scripts/monitor_training.sh`